In [0]:
silver_df = spark.table("silver_customer_products")

display(silver_df.limit(5))

In [0]:
print("Total rows")

print(silver_df.count())


In [0]:
print("Distinct orders")

print(
    silver_df
    .select("order_id")
    .distinct()
    .count()
)

In [0]:
print("Min Order ID")

display(
    silver_df.selectExpr(
        "min(order_id)"
    )
)


print("Max Order ID")

display(
    silver_df.selectExpr(
        "max(order_id)"
    )
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.orderBy("order_id")

orders_batch = (
    silver_df
    .select("order_id")
    .distinct()
    .orderBy("order_id")
    .withColumn(
        "row_num",
        row_number().over(window)
    )
    .withColumn(
        "batch_no",
        ((row_number().over(window)-1)/1000).cast("int")
    )
)

display(orders_batch.limit(10))

In [0]:
orders_batch.selectExpr(
    "max(batch_no)"
).show()

In [0]:
stream_path = "/Volumes/workspace/default/e-commerce_recommendation/incoming_orders"

In [0]:
batch0_ids = (
    orders_batch
    .filter("batch_no = 0")
    .select("order_id")
)

batch0 = (
    silver_df
    .join(
        batch0_ids,
        on="order_id"
    )
)

display(batch0.limit(10))

In [0]:
batch0.write.mode("overwrite").csv(

    f"{stream_path}/batch_0"

)

print("Batch 0 sent")

In [0]:
dbutils.fs.rm(stream_path,True)
dbutils.fs.mkdirs(stream_path)

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_000.csv"
    )
)

In [0]:
import time

for i in range(15,20):

    ids = (
        orders_batch
        .filter(f"batch_no={i}")
        .select("order_id")
    )

    batch = silver_df.join(ids,"order_id")


    temp_path = f"/Volumes/workspace/default/e-commerce_recommendation/temp"


    dbutils.fs.rm(temp_path,True)


    (
        batch
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header",True)
        .csv(temp_path)
    )


    files = dbutils.fs.ls(temp_path)

    csv_file = [f.path for f in files if f.name.endswith(".csv")][0]


    target = f"{stream_path}/orders_{i:03d}.csv"


    dbutils.fs.mv(
        csv_file,
        target
    )


    print(target)

    time.sleep(5)